In [1]:
# NAME: Bruce Bai
# GITHUB USERNAME: bruceb002
# USC ID: 1563103829

import pandas as pd
from pathlib import Path
import os
import glob
import numpy as np

In [2]:
# 1a) - 1c)ii. All parts combined into a single loop to avoid opening files multiple times

p = Path(r"../data/AReM")
the_bendings = set(["bending1", "bending2"])

all_instances_features = []
labels = []

# tracks train/test splits, file names, etc.
instance_metadata = [] 

for f in p.iterdir():
    if f.is_dir():
        if f.name in the_bendings:
            test_nums = {1, 2}
        else:
            test_nums = {1, 2, 3}
            
        for csv_file in f.glob('*.csv'):
            curr_file = csv_file.stem
            file_num = int(''.join(filter(str.isdigit, curr_file)))

            # Apply 1b Logic: is_test Variable used later to determine Train vs Test data set
            is_test = file_num in test_nums

            # 1c)ii: Feature Extraction
            try:
                df = pd.read_csv(csv_file, comment='#', sep=r'\s+', header=None, on_bad_lines='skip')
                if df.shape[1] <= 1:
                    df = pd.read_csv(csv_file, comment='#', sep=',', header=None, on_bad_lines='skip')
                if df.shape[1] <= 1:
                    df = pd.read_csv(csv_file, comment='#', sep=None, engine='python', header=None, on_bad_lines='skip')
            except Exception:
                df = pd.read_csv(csv_file, comment='#', sep=None, engine='python', header=None, on_bad_lines='skip')

            # Guard check: If it still only has 1 column, let's print the filename so we can see which file is causing it!
            if df.shape[1] < 7:
                print(f"Skipping or inspecting malformed file: {csv_file.name} (Columns found: {df.shape[1]})")
                continue  # skip this file if it's unparseable

            # Dynamically trim to the first 7 columns and assign names safely
            df = df.iloc[:, :7]
            df.columns = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
            sensor_data = df.drop(columns=['time'])
            
            # Extract the 7 requested time-domain features for each of the 6 time series
            file_features = []
            for col in sensor_data.columns:
                s = sensor_data[col]
                file_features.extend([
                    s.min(),
                    s.max(),
                    s.mean(),
                    s.median(),
                    s.std(),
                    s.quantile(0.25),
                    s.quantile(0.75)
                ])
            
            # Appends once per file
            all_instances_features.append(file_features)
            labels.append(f.name)
            instance_metadata.append({'file': csv_file.name, 'activity': f.name, 'is_test': is_test})

# 1c)ii. Convert to the 88-row master feature matrix
#        following structural/naming conventions as closely as possible
feature_names = []
for i, col in enumerate(sensor_data.columns, start=1):
    feature_names.extend([
        f'min_{i}', 
        f'max_{i}', 
        f'mean_{i}', 
        f'median_{i}', 
        f'std_{i}', 
        f'1st quart_{i}', 
        f'3rd quart_{i}'
    ])
    
# Ensure exact 1 --> 88 row indexing
master_df = pd.DataFrame(all_instances_features, columns=feature_names)
master_df.index = range(1, len(master_df) + 1)  # Set index from 1 to 88 inclusive
master_df.index.name = 'Instance'

# Keep labels and is_test as separate companion arrays/lists for training/testing later
labels_array = np.array(labels)
is_test_array = np.array([m['is_test'] for m in instance_metadata])

# Using part 1b logic: Explicitly assign train/test splits
X_train = master_df[~is_test_array]
X_test = master_df[is_test_array]
y_train = labels_array[~is_test_array]
y_test = labels_array[is_test_array]

# Output shapes to confirm
print("Strict feature matrix shape:", master_df.shape)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("")

#Print the master dataframe
print(master_df)

# Resources found for this part
# https://www.geeksforgeeks.org/python/python-loop-through-folders-and-files-in-directory/

Strict feature matrix shape: (88, 42)
Training set shape: (69, 42)
Testing set shape: (19, 42)

          min_1  max_1     mean_1  median_1     std_1  1st quart_1  \
Instance                                                             
1         36.25  48.00  43.969125     44.50  1.618364       43.310   
2         37.00  48.00  43.454958     43.25  1.386098       42.500   
3         33.00  47.75  42.179812     43.50  3.670666       39.150   
4         33.00  45.75  41.678063     41.75  2.243490       41.330   
5         37.25  45.00  40.624792     40.50  1.476967       39.250   
...         ...    ...        ...       ...       ...          ...   
84        18.50  44.25  35.752354     36.00  4.614802       33.000   
85        24.25  45.00  37.177042     36.25  3.581301       34.500   
86        23.33  43.50  36.248768     36.75  3.824632       33.415   
87        26.25  44.25  36.957458     36.29  3.434863       34.500   
88        27.75  44.67  37.142359     36.33  3.762442       34.0

### c)i. Types of Time-Domain Features Used in Time-Series Classification
Time-domain features are basically stats and metrics calculated straight from the raw signal values as they change over time. Some of the main categories include:
- **Central Tendency:** Mean and median, which give you a sense of the baseline or average level of the signal.
- **Spread & Dispersion:** Standard deviation, variance, range, and interquartile range (IQR). These show how much the signal is bouncing around or fluctuating.
- **Extrema:** Minimum and maximum values to track the absolute lower and upper bounds.
- **Distribution Shape:** Skewness and kurtosis, which measure whether the data is lopsided or has crazy sharp peaks/outliers.
- **Energy:** Root Mean Square (RMS), which captures the overall power or energy of the waveform.
- **Temporal Dependency:** Autocorrelation, which looks at how a data point relates to its past lagged values over time.

### Resources found
\# https://www.geeksforgeeks.org/data-analysis/feature-engineering-for-time-series-data-methods-and-applications/

In [3]:
# c)iii. Obtaining Bootstrap Confidence Interval

bootstrap_results = []
n_bootstraps = 1000

# 90% confidence interval (1 - 0.10)
alpha = 0.10  

# Set a seed for reproducibility
np.random.seed(1313)

for col in master_df.columns:
    feature_values = master_df[col].values
    
    # 1. Estimate sample standard deviation across the 88 instances
    sample_std = np.std(feature_values, ddof=1)
    
    # 2. Bootstrap resampling to build the 90% confidence interval
    boot_stds = []
    for _ in range(n_bootstraps):
        boot_sample = np.random.choice(feature_values, size=len(feature_values), replace=True)
        boot_stds.append(np.std(boot_sample, ddof=1))
        
    # Calculate 90% Confidence Interval bounds (5th and 95th percentiles)
    ci_lower_bound = np.percentile(boot_stds, (alpha / 2) * 100)
    ci_upper_bound = np.percentile(boot_stds, (1 - alpha / 2) * 100)
    
    bootstrap_results.append({
        'Feature': col,
        'Sample Std': sample_std,
        '90% CI Lower (5%)': ci_lower_bound,
        '90% CI Upper (95%)': ci_upper_bound
    })

# Convert to a summary dataframe
bootstrap_summary_df = pd.DataFrame(bootstrap_results)

# Display the summary table for all 42 features
print("90% Bootstrap Confidence Intervals for Feature Standard Deviations: \n")
print(bootstrap_summary_df.to_string(index=False))

90% Bootstrap Confidence Intervals for Feature Standard Deviations: 

    Feature  Sample Std  90% CI Lower (5%)  90% CI Upper (95%)
      min_1    9.569975           8.268968           10.780150
      max_1    4.394362           3.357585            5.324679
     mean_1    5.335700           4.656371            5.910902
   median_1    5.440054           4.813759            6.031982
      std_1    1.772185           1.573067            1.956729
1st quart_1    6.153874           5.580469            6.625791
3rd quart_1    5.138925           4.309877            5.864568
      min_2    0.000000           0.000000            0.000000
      max_2    5.062729           4.638102            5.400841
     mean_2    1.574198           1.400166            1.705552
   median_2    1.412293           1.234096            1.550463
      std_2    0.884137           0.811813            0.942749
1st quart_2    0.946386           0.831978            1.036354
3rd quart_2    2.125399           1.870091      

In [4]:
# c)iv. Self-intuitive best three time-domain features

print("To capture the core characteristics of the time-series data without overcomplicating things, \
we can select mean, standard deviation, and maximum as our three representative time-domain features \
(following the prompt's suggested statistical breakdown).\n")

print("The mean gives us the baseline signal level or average intensity of the movement for each sensor,\n\
the standard deviation captures how much the sensor values fluctuate and bounce around during each activity instance,\n\
and the maximum points to the peak intensity or highest spike reached during the action.\n")

print("Instead of doing a brute-force empirical search at this exploratory stage, this trio gives us a clean, \
well-rounded summary of the signal's core behavior—covering where it centers, how volatile it is, and where it peaks.\n")



To capture the core characteristics of the time-series data without overcomplicating things, we can select mean, standard deviation, and maximum as our three representative time-domain features (following the prompt's suggested statistical breakdown).

The mean gives us the baseline signal level or average intensity of the movement for each sensor,
the standard deviation captures how much the sensor values fluctuate and bounce around during each activity instance,
and the maximum points to the peak intensity or highest spike reached during the action.

Instead of doing a brute-force empirical search at this exploratory stage, this trio gives us a clean, well-rounded summary of the signal's core behavior—covering where it centers, how volatile it is, and where it peaks.

